# signVLM training (Google Colab)

This notebook follows the **SignVLM** stack in this repository (`main.py`, `model.py`, `video_dataset/`): CLIP ViT backbone + temporal decoder + classifier, **AdamW**, **CosineAnnealingLR**, **CrossEntropyLoss**, **AMP fp16**, and list files in the form `path<TAB>label` (see `SIGNVLM_TRAINING_NOTES.md` and `docs/signVLMPipeline.md`).

**Cell 1 — Setup:** installs PyAV (`av`), **scikit-learn** (confusion matrix + F1), **pillow**, and prints PyTorch/CUDA.

**Cell 2** mounts Drive and sets `BASE`, checkpoints, list dir, and default **CLIP weights** on Drive (`DRIVE_REPO_ROOT/CLIP_weights/...`). **Cell 2b** loads training **Python code** from **GitHub** (`SIGNVLM_GIT_URL` + `SIGNVLM_GIT_BRANCH`) into `/content/...`, or uses a full repo already on Drive under `DRIVE_REPO_ROOT` if `main.py` exists there. **Cell 2c** (Colab) copies `train_data` / `validation_data` / `test_data` from Drive to `/content/signvlm_local_data` for fast DataLoader I/O.

In [1]:
# Cell 1: Setup and installations
import subprocess
import sys

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install(["av", "pillow", "scikit-learn"])
print("OK: av (PyAV), pillow, scikit-learn")

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

OK: av (PyAV), pillow, scikit-learn
torch: 2.10.0+cu128 cuda: True


## Cell 2: Mount Drive and paths

On **Colab**, `BASE` should live under **`/content/drive/MyDrive/...`** so **checkpoints** (`signVLM_checkpoints/`) and **list files** (`signVLM_lists/`) persist on **Google Drive**, not ephemeral `/content`.

**Overrides:** set `SIGNVLM_DRIVE_BASE` (full path to your dataset root), `SIGNVLM_BACKBONE_PATH` (ViT-L `.pt`), or optional `SIGNVLM_TRAIN_SUBDIR` / `VAL` / `TEST` / `UNSEEN` / `REPO_SUBDIR`. Off Colab, use `SIGNVLM_BASE` or rely on `./signvlm_data`.

**`WORK_ROOT`** defaults to `/content`. **`CODE_ROOT`** is set in **Cell 2b** (git clone path or `DRIVE_REPO_ROOT`). Default **ViT weights** use `DRIVE_REPO_ROOT/CLIP_weights/...` on Drive (override with `SIGNVLM_BACKBONE_PATH`).

Expected layout under `BASE`:

- `train_data/<class_name>/*.mp4` (or frames under `<class>/<stem>/` if using `frames_available=1`)
- `validation_data/...`
- `test_data/...`
- `Unseen_data/...` (optional; Cell 4 writes an empty `unseen.tsv` if missing)

**Cell 2b (training code):** set **`SIGNVLM_GIT_URL`** to your public or token-based clone URL (e.g. run `import os; os.environ['SIGNVLM_GIT_URL'] = 'https://github.com/org/repo.git'` in a cell before 2b). Optional: **`SIGNVLM_GIT_BRANCH`**, **`SIGNVLM_GIT_DEPTH`**, **`SIGNVLM_GIT_DIR`**, **`SIGNVLM_GIT_RECLONE=1`**. If unset and **`DRIVE_REPO_ROOT/main.py`** exists, the Drive copy is used. **Private repos:** token-in-URL or Colab **Secrets** (error text in 2b). **Weights** default to **`DRIVE_REPO_ROOT/CLIP_weights/...`** in Cell 2, not the git tree.

**Cell 2c** (recommended on Colab): copies **video/frame folders** to **`/content/signvlm_local_data/`** so training I/O is local; checkpoints and TSV lists stay on Drive (Cell 2). Override with `SIGNVLM_STAGE_TO_LOCAL=0` to read media directly from `BASE`.

In [2]:
# Cell 2: Mount Google Drive and directory paths
# Checkpoints go to CHECKPOINT_DIR; on Colab keep BASE under Drive so artifacts survive session loss.
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_ROOT = "/content"
CODE_ROOT = f"{WORK_ROOT}/signvlm_bundle"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MY = "/content/drive/MyDrive"
else:
    DRIVE_MY = os.environ.get("SIGNVLM_DRIVE_MY", "")

BASE = os.environ.get("SIGNVLM_DRIVE_BASE", "").strip()
if not BASE:
    if IN_COLAB and DRIVE_MY:
        BASE = f"{DRIVE_MY}/FYP_DATA"
    else:
        BASE = os.environ.get("SIGNVLM_BASE", os.path.join(os.getcwd(), "signvlm_data"))

TRAIN_SUBDIR = os.environ.get("SIGNVLM_TRAIN_SUBDIR", "train_data")
VAL_SUBDIR = os.environ.get("SIGNVLM_VAL_SUBDIR", "validation_data")
TEST_SUBDIR = os.environ.get("SIGNVLM_TEST_SUBDIR", "test_data")
UNSEEN_SUBDIR = os.environ.get("SIGNVLM_UNSEEN_SUBDIR", "Unseen_data")

TRAIN_DIR = f"{BASE}/{TRAIN_SUBDIR}"
VAL_DIR = f"{BASE}/{VAL_SUBDIR}"
TEST_DIR = f"{BASE}/{TEST_SUBDIR}"
UNSEEN_DIR = f"{BASE}/{UNSEEN_SUBDIR}"

REPO_SUBDIR = os.environ.get("SIGNVLM_REPO_SUBDIR", "Urdu-Sign-Language-Recognition-using-SignVLM")
DRIVE_REPO_ROOT = f"{BASE}/{REPO_SUBDIR}"
REPO_ROOT = DRIVE_REPO_ROOT

BACKBONE_PATH = os.environ.get("SIGNVLM_BACKBONE_PATH", "").strip()
if not BACKBONE_PATH:
    BACKBONE_PATH = f"{DRIVE_REPO_ROOT}/CLIP_weights/ViT-L/ViT-L/ViT-L-14.pt"

CHECKPOINT_DIR = f"{BASE}/signVLM_checkpoints"
LIST_DIR = f"{BASE}/signVLM_lists"

if IN_COLAB and not CHECKPOINT_DIR.startswith("/content/drive/MyDrive"):
    raise RuntimeError(
        "CHECKPOINT_DIR must be under mounted Google Drive (/content/drive/MyDrive/...). "
        "Set SIGNVLM_DRIVE_BASE to a path under MyDrive, not /content alone."
    )

if not os.path.isfile(BACKBONE_PATH):
    raise FileNotFoundError(
        f"CLIP backbone not found: {BACKBONE_PATH}\n"
        "Set SIGNVLM_BACKBONE_PATH or place ViT-L-14.pt under DRIVE_REPO_ROOT/CLIP_weights/... on Drive."
    )

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LIST_DIR, exist_ok=True)
print(
    f"IN_COLAB={IN_COLAB} WORK_ROOT={WORK_ROOT} CODE_ROOT={CODE_ROOT} BASE={BASE} "
    f"DRIVE_REPO_ROOT={DRIVE_REPO_ROOT} REPO_ROOT={REPO_ROOT} (overridden in 2b if using git) "
    f"BACKBONE_PATH={BACKBONE_PATH} CHECKPOINT_DIR={CHECKPOINT_DIR} LIST_DIR={LIST_DIR}"
)

Mounted at /content/drive
IN_COLAB=True WORK_ROOT=/content CODE_ROOT=/content/signvlm_bundle BASE=/content/drive/MyDrive/FYP_DATA REPO_ROOT=/content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM BACKBONE_PATH=/content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM/CLIP_weights/ViT-L/ViT-L/ViT-L-14.pt CHECKPOINT_DIR=/content/drive/MyDrive/FYP_DATA/signVLM_checkpoints LIST_DIR=/content/drive/MyDrive/FYP_DATA/signVLM_lists


In [ ]:
# Cell 2b: Training code from GitHub (preferred) or a full repo copy on Drive
import os
import shutil
import subprocess

WORK_ROOT = globals().get("WORK_ROOT", "/content")
CODE_ROOT = globals().get("CODE_ROOT", f"{WORK_ROOT}/signvlm_bundle")
DRIVE_REPO_ROOT = globals().get("DRIVE_REPO_ROOT", "")
REPO_ROOT = globals().get("REPO_ROOT", DRIVE_REPO_ROOT)

GIT_URL = os.environ.get("SIGNVLM_GIT_URL", "").strip()
GIT_BRANCH = (os.environ.get("SIGNVLM_GIT_BRANCH", "main") or "main").strip()
try:
    GIT_DEPTH = int(os.environ.get("SIGNVLM_GIT_DEPTH", "1") or "1")
except ValueError:
    GIT_DEPTH = 1
GIT_DIR = os.environ.get("SIGNVLM_GIT_DIR", "").strip() or os.path.join(
    WORK_ROOT, "Urdu-Sign-Language-Recognition-using-SignVLM"
)
RECLONE = os.environ.get("SIGNVLM_GIT_RECLONE", "").strip().lower() in ("1", "true", "yes")


def _has_main(path: str) -> bool:
    return bool(path) and os.path.isfile(os.path.join(path, "main.py"))


if GIT_URL:
    if RECLONE and os.path.isdir(GIT_DIR):
        shutil.rmtree(GIT_DIR, ignore_errors=True)
    if os.path.isdir(os.path.join(GIT_DIR, ".git")):
        subprocess.run(
            ["git", "-C", GIT_DIR, "fetch", "--all", "--prune"],
            check=False,
        )
        subprocess.run(["git", "-C", GIT_DIR, "checkout", GIT_BRANCH], check=True)
        subprocess.run(
            ["git", "-C", GIT_DIR, "pull", "--ff-only"],
            check=False,
        )
    else:
        if os.path.exists(GIT_DIR):
            shutil.rmtree(GIT_DIR, ignore_errors=True)
        parent = os.path.dirname(GIT_DIR)
        if parent:
            os.makedirs(parent, exist_ok=True)
        cmd = ["git", "clone"]
        if GIT_DEPTH > 0:
            cmd.extend(["--depth", str(GIT_DEPTH)])
        cmd.extend(["-b", GIT_BRANCH, GIT_URL, GIT_DIR])
        print("Running:", " ".join(cmd))
        subprocess.run(cmd, check=True)
    REPO_ROOT = GIT_DIR
    CODE_ROOT = GIT_DIR
    print("Using SignVLM from git:", CODE_ROOT, "branch:", GIT_BRANCH)
elif _has_main(DRIVE_REPO_ROOT):
    REPO_ROOT = DRIVE_REPO_ROOT
    CODE_ROOT = DRIVE_REPO_ROOT
    print("Using SignVLM from Drive (DRIVE_REPO_ROOT):", CODE_ROOT)
else:
    _msg = (
        "No training code found. Either set environment variable SIGNVLM_GIT_URL to your "
        "GitHub clone URL and re-run this cell, or place a full repo with main.py under:\n  "
        + DRIVE_REPO_ROOT
        + "\nFor private repos, use: https://<token>@github.com/<org>/<repo>.git (Colab: store token in User secrets).\n"
        "Optional: SIGNVLM_GIT_BRANCH (default main), SIGNVLM_GIT_DEPTH (default 1), "
        "SIGNVLM_GIT_DIR (clone path), SIGNVLM_GIT_RECLONE=1 to delete and re-clone."
    )
    raise FileNotFoundError(_msg)


## Cell 2c: Stage dataset to Colab local disk (recommended)

Training touches **many files per step**. Reading **`train_data` / `validation_data` / `test_data`** from **Google Drive** is high-latency; copying those trees once into **`/content/signvlm_local_data/...`** (VM local SSD) usually improves throughput and stabilizes step times.

- **Checkpoints** (`CHECKPOINT_DIR`) and **lists** (`LIST_DIR`) stay on Drive (Cell 2). Python training code is from **Cell 2b** (git or `DRIVE_REPO_ROOT`).
- **`SIGNVLM_STAGE_TO_LOCAL`**: unset or `1` on Colab → copy splits to `/content` before training. Set to `0` to read media directly from `BASE` (slower on Drive).
- **`SIGNVLM_REUSE_LOCAL_STAGE=1`**: if the local staging folder already has data, **skip** re-copying (useful when re-running downstream cells in the same session).
- **Progress:** each split prints an **ASCII tqdm bar to stdout** (plain console text, not the Jupyter/HTML tqdm widget). With **`rsync`**, the bar follows **`--info=progress2`**; if `rsync` is missing or errors, a **per-file** text bar is used for the shutil copy fallback.

Run **Cell 2c** after Cell 2 and 2b, then continue with Cell 3 onward.


In [4]:
# Cell 2c: Copy train/val/test splits from Drive to /content for fast DataLoader I/O
import os
import re
import shutil
import subprocess
import sys
import time

try:
    from tqdm.std import tqdm
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "tqdm"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    from tqdm.std import tqdm  # type: ignore

# Text console progress (ASCII bar on stdout), not the Jupyter/HTML tqdm widget.
_TQDM = {"file": sys.stdout, "ascii": True, "dynamic_ncols": False, "ncols": 96, "mininterval": 0.25, "leave": True}


def _parse_bool_env(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if not raw:
        return default
    return raw in ("1", "true", "yes", "on")


WORK_ROOT = globals().get("WORK_ROOT", "/content")
IN_COLAB = globals().get("IN_COLAB", False)
TRAIN_SUBDIR = globals().get("TRAIN_SUBDIR", "train_data")
VAL_SUBDIR = globals().get("VAL_SUBDIR", "validation_data")
TEST_SUBDIR = globals().get("TEST_SUBDIR", "test_data")

stage = _parse_bool_env("SIGNVLM_STAGE_TO_LOCAL", default=IN_COLAB)
reuse = _parse_bool_env("SIGNVLM_REUSE_LOCAL_STAGE", default=False)

SIGNVLM_DATA_IS_LOCAL = False

train_src = globals()["TRAIN_DIR"]
val_src = globals()["VAL_DIR"]
test_src = globals()["TEST_DIR"]


def _copytree_with_progress(src: str, dst: str, desc: str) -> None:
    """Fallback when rsync is missing or fails: copy files with a text tqdm bar (per file)."""
    src = os.path.abspath(src)
    pairs = []
    for root, _dirs, files in os.walk(src):
        for name in files:
            sp = os.path.join(root, name)
            pairs.append((sp, os.path.relpath(sp, src)))
    for sp, rel in tqdm(pairs, desc=desc, unit="file", **_TQDM):
        dp = os.path.join(dst, rel)
        os.makedirs(os.path.dirname(dp), exist_ok=True)
        shutil.copy2(sp, dp)


def _rsync_with_progress(src: str, dst: str, desc: str) -> None:
    """rsync -a; tqdm bar is driven by --info=progress2 lines (transfer % on stderr)."""
    rsync = shutil.which("rsync")
    if not rsync:
        raise FileNotFoundError("rsync not found")
    os.makedirs(dst, exist_ok=True)
    pct_re = re.compile(r"(\d+)%")
    proc = subprocess.Popen(
        [
            rsync,
            "-a",
            "--info=progress2",
            f"{src.rstrip(os.sep)}/",
            f"{dst.rstrip(os.sep)}/",
        ],
        stderr=subprocess.PIPE,
        stdout=subprocess.DEVNULL,
        text=True,
        bufsize=1,
    )
    assert proc.stderr is not None
    with tqdm(
        total=100,
        desc=desc,
        unit="%",
        bar_format="{desc}: {n:3d}%|{bar}| {elapsed}",
        **_TQDM,
    ) as bar:
        for line in iter(proc.stderr.readline, ""):
            m = pct_re.search(line)
            if m:
                bar.n = min(100, int(m.group(1)))
                bar.refresh()
        proc.wait()
        if proc.returncode != 0:
            raise subprocess.CalledProcessError(proc.returncode, rsync)
        bar.n = 100
        bar.refresh()


def _sync_split(label: str, src: str, dst: str) -> None:
    if not os.path.isdir(src):
        raise FileNotFoundError(f"{label}: missing source directory: {src}")
    if reuse and os.path.isdir(dst) and os.listdir(dst):
        print(f"{label}: reuse existing local copy -> {dst}")
        return
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    parent = os.path.dirname(dst.rstrip(os.sep))
    if parent:
        os.makedirs(parent, exist_ok=True)
    t0 = time.perf_counter()
    rsync = shutil.which("rsync")
    desc = f"{label} Drive to /content"
    if rsync:
        try:
            _rsync_with_progress(src, dst, desc)
        except (subprocess.CalledProcessError, FileNotFoundError, BrokenPipeError):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            os.makedirs(dst, exist_ok=True)
            _copytree_with_progress(src, dst, desc + " (file copy)")
    else:
        os.makedirs(dst, exist_ok=True)
        _copytree_with_progress(src, dst, desc)
    print(f"{label}: staged in {time.perf_counter() - t0:.1f}s -> {dst}")


if stage and IN_COLAB:
    local_root = os.path.join(WORK_ROOT, "signvlm_local_data")
    train_dst = os.path.join(local_root, TRAIN_SUBDIR)
    val_dst = os.path.join(local_root, VAL_SUBDIR)
    test_dst = os.path.join(local_root, TEST_SUBDIR)
    _sync_split("train", train_src, train_dst)
    _sync_split("validation", val_src, val_dst)
    _sync_split("test", test_src, test_dst)
    TRAIN_DIR, VAL_DIR, TEST_DIR = train_dst, val_dst, test_dst
    SIGNVLM_DATA_IS_LOCAL = True
    print("SIGNVLM_DATA_IS_LOCAL=True")
    print("TRAIN_DIR:", TRAIN_DIR)
    print("VAL_DIR:", VAL_DIR)
    print("TEST_DIR:", TEST_DIR)
elif stage and not IN_COLAB:
    print("SIGNVLM_STAGE_TO_LOCAL set but IN_COLAB=False; no auto-copy. Paths unchanged.")
else:
    print("Local staging off (SIGNVLM_STAGE_TO_LOCAL=0 or default off). Using Cell 2 paths:")
    print("TRAIN_DIR:", train_src)


train Drive to /content:   0%|                                                           | 00:00

train Drive to /content:   0%|                                                           | 03:36


KeyboardInterrupt: 

## Cell 3: Imports

Run **Cell 2** (Drive paths + `DRIVE_REPO_ROOT` + `BACKBONE_PATH` on Drive), **Cell 2b** (git clone or Drive `main.py`), then **Cell 2c** (optional staging to `/content`; skip if `SIGNVLM_STAGE_TO_LOCAL=0`). After 2b, `CODE_ROOT` is the **git** checkout or your Drive repo. Initializes **single-GPU distributed** like `main.py` (FileStore + `gloo`).

In [ ]:
# Cell 3: Imports and single-process distributed init (matches main.py)
# Run Cell 2, Cell 2b, then Cell 2c (staging) — training code lives in CODE_ROOT (Cell 2 / 2b).
import os
import sys
import tempfile
import atexit

CODE_ROOT = globals().get("CODE_ROOT", "/content/signvlm_bundle")

if not os.path.isfile(os.path.join(CODE_ROOT, "main.py")):
    raise FileNotFoundError(
        "SignVLM sources not found. Run Cell 2b (set SIGNVLM_GIT_URL or place main.py under DRIVE_REPO_ROOT). Expected: "
        + os.path.join(CODE_ROOT, "main.py")
    )

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)
os.chdir(CODE_ROOT)

import torch
import torch.distributed as dist

if not dist.is_initialized():
    _store_path = os.path.join(tempfile.gettempdir(), f"signvlm_store_{os.getpid()}")
    _store = dist.FileStore(_store_path, 1)
    dist.init_process_group(backend="gloo", store=_store, rank=0, world_size=1)
    atexit.register(lambda p=_store_path: os.remove(p) if os.path.exists(p) else None)

import checkpoint
import video_dataset
from model import EVLTransformer
from weight_loaders import weight_loader_fn_dict
from vision_transformer import vit_presets

print("Code (sys.path):", CODE_ROOT)
print("REPO_ROOT (Cell 2 paths / CLIP weights):", globals().get("REPO_ROOT", "<run Cell 2 for Drive paths>"))
print("CUDA:", torch.cuda.is_available())


Code (sys.path): /content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM
REPO_ROOT (Cell 2 paths / CLIP weights): /content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM
CUDA: True


### Confusion matrix & F1 helpers

`collect_predictions` matches `main.evaluate` (batch size 1, multi-view softmax mean). `print_confusion_and_f1` prints macro/weighted/micro F1, `classification_report`, and the confusion matrix (or saves `.npy` when `num_classes` is large).

In [ ]:
# Metrics (same inference as main.evaluate)
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score


def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data, labels in loader:
            data = data.cuda(non_blocking=True)
            labels = labels.cuda(non_blocking=True)
            assert data.size(0) == 1
            if data.ndim == 6:
                data = data[0]
            logits = model(data)
            scores = logits.softmax(dim=-1).mean(dim=0)
            y_pred.append(int(scores.argmax().cpu()))
            y_true.append(int(labels.view(-1)[0].cpu()))
    return np.array(y_true), np.array(y_pred)


def print_confusion_and_f1(y_true, y_pred, num_classes, class_names, title, save_dir=None, max_print_classes=35):
    labels_idx = np.arange(num_classes)
    cm = confusion_matrix(y_true, y_pred, labels=labels_idx)
    names = [class_names[i] if i < len(class_names) else str(i) for i in range(num_classes)]
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
    print(f"F1 (macro):     {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1 (weighted):  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1 (micro):     {f1_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
    print("\nClassification report:")
    print(
        classification_report(
            y_true, y_pred, labels=labels_idx, target_names=names, zero_division=0, digits=4
        )
    )
    if num_classes <= max_print_classes:
        print("\nConfusion matrix (rows=true class, cols=predicted class):")
        print(cm)
    else:
        print(f"\nConfusion matrix shape {cm.shape} (too large for full print).")
        if save_dir:
            import os
            safe = title.lower().replace(" ", "_").replace("/", "_")
            p = os.path.join(save_dir, f"{safe}_confusion_matrix.npy")
            np.save(p, cm)
            print(f"Saved full matrix: {p}")
        h = min(12, cm.shape[0])
        print(f"Top-left {h}x{h} block:\n{cm[:h, :h]}")

## Cell 4: Data preparation — TSV lists + loaders

- Builds a **class name → id** map from sorted folder names under `train_data` (ids `0 .. num_classes-1`).
- Writes `train.tsv`, `val.tsv`, `test.tsv`, and `unseen.tsv` (empty if `Unseen_data` is missing) with lines `relative_path<TAB>label`, paths relative to each split root (same convention as `prepare_psl_splits.py` when `data_root` is the split folder).
- **Videos:** `frames_available=0` (PyAV). For **pre-extracted frames**, set `FRAMES_AVAILABLE = 1` and use the repo’s `tools/extract_frames_signvlm.py` first; lists must use virtual `.mp4` paths as in `SIGNVLM_TRAINING_NOTES.md`.
- Adjust **`NUM_CLASSES`** if you use a fixed label map file instead (must match folder names / ids).

### DataLoader / Colab notes

- After **Cell 2c**, `TRAIN_DIR` / `VAL_DIR` / `TEST_DIR` point at **`/content/signvlm_local_data/...`**: use **`num_workers` 4–8** and `prefetch_factor` 2–4 (set in the code cell below). If media still lives on **Drive** (`SIGNVLM_STAGE_TO_LOCAL=0`), keep **`num_workers` 0–2**; extra workers often increase random I/O and slow things down.
- **`local_data_cache` (default on Colab when data is on Drive):** `video_dataset` copies each list item from the split root into `/content/signvlm_data_cache/...` on **first** access and reuses the local copy for **all later steps and epochs** in that session. Set `LOCAL_DATA_CACHE = ""` in the code cell to disable. `/content` is cleared when the runtime disconnects; checkpoints on Drive are unchanged.
- **I/O vs GPU:** set **`IO_SPEED_TEST_DUMMY = True`** in the code cell below to use `dummy_dataset` (fake tensors, same shapes) and compare printed step times to real data. A large gap means disk/decode/augment bound, not GPU matmul bound.
- **Future:** with `frames_available=0`, `video_dataset/dataset.py` currently **decodes whole videos** per sample. For long-term speed (especially if you skip staging), plan **partial PyAV decode / seek** so only the needed temporal span is decoded (see comment block in `dataset.py` next to `av.open`).

In [ ]:
# Cell 4: Build label map and TSV lists; create train/val DataLoaders via repo helpers
from pathlib import Path
import argparse
import json
import os

FRAMES_AVAILABLE = 0  # 1 if using pre-extracted frames (see SIGNVLM_TRAINING_NOTES.md)

# True = no real disk reads in train_loader (same tensor shapes); use to compare vs real data I/O.
IO_SPEED_TEST_DUMMY = False


def sorted_label_dirs(root: Path):
    return sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.name.casefold())


def build_label_map_from_train(train_root: Path):
    labels = [d.name for d in sorted_label_dirs(train_root)]
    return {name: i for i, name in enumerate(labels)}, labels


def iter_videos(label_dir: Path):
    vids = sorted(label_dir.glob("*.mp4"), key=lambda p: (not p.stem.isdigit(), int(p.stem) if p.stem.isdigit() else p.stem))
    for v in vids:
        yield v


def write_split_tsv(split_root: Path, rel_prefix: Path, name_to_id: dict, out_path: Path):
    n = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for label_dir in sorted_label_dirs(split_root):
            name = label_dir.name
            if name not in name_to_id:
                raise ValueError(f"Unknown label folder {name!r} under {split_root}; not in train label map")
            lid = name_to_id[name]
            for vid in iter_videos(label_dir):
                rel = (rel_prefix / name / vid.name).as_posix()
                f.write(f"{rel}\t{lid}\n")
                n += 1
    return n


def require_split_dir(path: str, split_label: str):
    if not os.path.isdir(path):
        raise FileNotFoundError(
            f"{split_label} directory missing: {path}\n"
            f"Expected under BASE={BASE} (see Cell 2). Create class subfolders with .mp4 files."
        )


require_split_dir(TRAIN_DIR, "train")
require_split_dir(VAL_DIR, "validation")
require_split_dir(TEST_DIR, "test")

train_root = Path(TRAIN_DIR)
name_to_id, class_names = build_label_map_from_train(train_root)
NUM_CLASSES = len(name_to_id)

label_json = Path(LIST_DIR) / "label_map_auto.json"
with open(label_json, "w", encoding="utf-8") as f:
    json.dump({str(i): class_names[i] for i in range(len(class_names))}, f, ensure_ascii=False, indent=2)
print(f"num_classes={NUM_CLASSES}, wrote {label_json}")

train_tsv = Path(LIST_DIR) / "train.tsv"
val_tsv = Path(LIST_DIR) / "val.tsv"
test_tsv = Path(LIST_DIR) / "test.tsv"
unseen_tsv = Path(LIST_DIR) / "unseen.tsv"

n_train = write_split_tsv(train_root, Path("."), name_to_id, train_tsv)
n_val = write_split_tsv(Path(VAL_DIR), Path("."), name_to_id, val_tsv)
n_test = write_split_tsv(Path(TEST_DIR), Path("."), name_to_id, test_tsv)

INCLUDE_UNSEEN = os.path.isdir(UNSEEN_DIR)
if INCLUDE_UNSEEN:
    n_unseen = write_split_tsv(Path(UNSEEN_DIR), Path("."), name_to_id, unseen_tsv)
else:
    unseen_tsv.write_text("", encoding="utf-8")
    n_unseen = 0
    print(f"INCLUDE_UNSEEN=False: missing {UNSEEN_DIR} - wrote empty {unseen_tsv}")

print(f"lines: train={n_train} val={n_val} test={n_test} unseen={n_unseen}")

_data_local = bool(globals().get("SIGNVLM_DATA_IS_LOCAL", False))
_work = globals().get("WORK_ROOT", "/content")
_in_c = globals().get("IN_COLAB", False)
# Cache-on-first-use: mirror list paths from train/val roots into local SSD; hits skip Drive for later epochs. Set to "" to disable.
LOCAL_DATA_CACHE = ""
if _in_c and not _data_local:
    LOCAL_DATA_CACHE = os.path.join(_work, "signvlm_data_cache")
    os.makedirs(LOCAL_DATA_CACHE, exist_ok=True)
_ncpu = os.cpu_count() or 4
if globals().get("IN_COLAB", False):
    if _data_local:
        _num_workers = min(8, max(4, _ncpu // 2))
        _prefetch_factor = 3
    else:
        _num_workers = min(2, max(0, (_ncpu // 4) or 1))
        _prefetch_factor = 2
else:
    _num_workers = min(4, _ncpu)
    _prefetch_factor = 3

# Namespace with fields expected by video_dataset + main (PSL-style hyperparameters from SIGNVLM_TRAINING_NOTES)
args = argparse.Namespace(
    frames_available=FRAMES_AVAILABLE,
    train_list_path=str(train_tsv),
    val_list_path=str(val_tsv),
    train_data_root=TRAIN_DIR,
    val_data_root=VAL_DIR,
    data_root="",
    local_data_cache=LOCAL_DATA_CACHE,
    batch_size=8,
    n_shots=-1,
    num_spatial_views=1,
    num_temporal_views=3,
    num_frames=24,
    sampling_rate=4,
    tsn_sampling=False,
    spatial_size=224,
    mean=[0.48145466, 0.4578275, 0.40821073],
    std=[0.26862954, 0.26130258, 0.27577711],
    num_workers=_num_workers,
    pin_memory=True,
    persistent_workers=_num_workers > 0,
    prefetch_factor=_prefetch_factor,
    dummy_dataset=IO_SPEED_TEST_DUMMY,
    auto_augment="rand-m7-n4-mstd0.5-inc1",
    interpolation="bicubic",
    mirror=True,
    num_steps=10000,
    checkpoint_dir=CHECKPOINT_DIR,
    auto_resume=False,
    resume_path=None,
    pretrain=None,
)

train_loader = video_dataset.create_train_loader(args, resume_step=0)
val_loader = video_dataset.create_val_loader(args)
print("train_loader steps:", len(train_loader), "expected:", args.num_steps)
print("val samples:", len(val_loader.dataset))
print(
    "DataLoader: num_workers=", args.num_workers,
    "prefetch_factor=", args.prefetch_factor,
    "dummy_dataset=", args.dummy_dataset,
    "SIGNVLM_DATA_IS_LOCAL=", _data_local,
    "local_data_cache=", getattr(args, "local_data_cache", "") or "(off)",
)


def make_eval_split_loader(tsv, root):
    # Eval-style loader (multi-view), same settings as val - train/test/unseen metrics.
    ns = argparse.Namespace(
        frames_available=args.frames_available,
        val_list_path=str(tsv),
        train_list_path=str(train_tsv),
        train_data_root=TRAIN_DIR,
        val_data_root=root,
        data_root="",
        local_data_cache=getattr(args, "local_data_cache", None) or "",
        batch_size=args.batch_size,
        n_shots=-1,
        num_spatial_views=1,
        num_temporal_views=3,
        num_frames=args.num_frames,
        sampling_rate=args.sampling_rate,
        tsn_sampling=False,
        spatial_size=args.spatial_size,
        mean=args.mean,
        std=args.std,
        num_workers=args.num_workers,
        pin_memory=getattr(args, "pin_memory", True),
        persistent_workers=getattr(args, "persistent_workers", args.num_workers > 0),
        prefetch_factor=getattr(args, "prefetch_factor", 3),
        dummy_dataset=False,
    )
    return video_dataset.create_val_loader(ns)


num_classes=104, wrote /content/drive/MyDrive/FYP_DATA/signVLM_lists/label_map_auto.json
INCLUDE_UNSEEN=False: missing /content/drive/MyDrive/FYP_DATA/Unseen_data - wrote empty /content/drive/MyDrive/FYP_DATA/signVLM_lists/unseen.tsv
lines: train=1556 val=312 test=101 unseen=0
1556
312
train_loader steps: 10000 expected: 10000
val samples: 312
num_workers: 2


## Cell 5: Model — `EVLTransformer` (signVLM)

Matches **PSL** script settings: **ViT-L/14-lnpre**, decoder **4×1024**, **16 heads**, temporal conv / pos / cross-attention enabled (`docs/signVLMPipeline.md`). Backbone defaults to **frozen fp16** unless you set `finetune_backbone=True`.

In [ ]:
# Cell 5: EVLTransformer (ViT-L/14 + decoder)
finetune_backbone = False  # set True for end-to-end (see docs/signVLMPipeline.md)
fp16 = True

model = EVLTransformer(
    backbone_name="ViT-L/14-lnpre",
    backbone_type="clip",
    backbone_path=BACKBONE_PATH,
    backbone_mode="finetune" if finetune_backbone else ("freeze_fp16" if fp16 else "freeze_fp32"),
    decoder_num_layers=4,
    decoder_qkv_dim=1024,
    decoder_num_heads=16,
    decoder_mlp_factor=4.0,
    num_classes=NUM_CLASSES,
    enable_temporal_conv=True,
    enable_temporal_pos_embed=True,
    enable_temporal_cross_attention=True,
    cls_dropout=0.5,
    decoder_mlp_dropout=0.5,
    num_frames=args.num_frames,
)
model.cuda()
print(model.__class__.__name__, "num_classes:", NUM_CLASSES)

---- backbone_path:  ViT-L/14-lnpre
EVLTransformer num_classes: 104


## Cell 6: Training configuration

- **Optimizer:** AdamW, `lr=4e-5`, `weight_decay=0.05` (`main.py` / notes).
- **Scheduler:** `CosineAnnealingLR(T_max=num_steps)`.
- **Loss:** `CrossEntropyLoss`.
- **AMP:** `GradScaler` + `autocast` (disable with `fp16=False`).
- **`batch_split=2`** matches the PSL shell script (gradient accumulation for memory).

In [ ]:
# Cell 6: Optimizer, scheduler, loss, AMP
lr = 4e-5
weight_decay = 0.05
batch_split = 2
num_steps = args.num_steps
save_freq = 2500
eval_freq = 2500
# Set to 0 to skip periodic confusion/F1 during training (keeps end-of-training metrics).
confusion_eval_freq = 0
print_freq = 10

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_steps)
loss_scaler = torch.amp.GradScaler("cuda", enabled=fp16)
criterion = torch.nn.CrossEntropyLoss()

args.save_freq = save_freq
args.eval_freq = eval_freq
args.print_freq = print_freq
args.confusion_eval_freq = confusion_eval_freq
args.batch_split = batch_split
args.fp16 = fp16
args.finetune_backbone = finetune_backbone

resume_step = checkpoint.resume_from_checkpoint(model, optimizer, lr_sched, loss_scaler, args)
print("resume_step:", resume_step)

Not resuming from a checkpoint.
resume_step: 0


## Cell 7: Training and validation loop

Mirrors `main.py`: micro-batching with `batch_split`, `GradScaler`, cosine LR step each iteration, periodic **`evaluate()`** (top-1 / top-5) on `val_loader`, and optional periodic **confusion matrix + F1** (controlled by `confusion_eval_freq`). After training finishes, runs confusion/F1 on the **train** split (eval-style loader). Checkpoints go to **Google Drive** (`CHECKPOINT_DIR`).

CLI alternative: `python main.py --train_list_path ... --val_list_path ... --checkpoint_dir ...` (see `scripts/train_psl_vitl14_16f_dec4x1024_1shot.sh`).

In [ ]:
# Cell 7: Training + validation (same structure as main.py)
from datetime import datetime

import main as signvlm_main

train_loader = video_dataset.create_train_loader(args, resume_step=resume_step)
assert len(train_loader) == args.num_steps - resume_step

model.train()
train_st = datetime.now()
batch_st = datetime.now()

for i, (data, labels) in enumerate(train_loader, resume_step):
    data = data.cuda(non_blocking=True)
    labels = labels.cuda(non_blocking=True)
    data_ed = datetime.now()
    optimizer.zero_grad(set_to_none=True)

    assert data.size(0) % args.batch_split == 0
    split_size = data.size(0) // args.batch_split
    hit1, hit5, loss_value = 0, 0, 0.0

    for j in range(args.batch_split):
        sl = slice(split_size * j, split_size * (j + 1))
        data_slice, labels_slice = data[sl], labels[sl]
        with torch.amp.autocast("cuda", enabled=args.fp16):
            logits = model(data_slice)
            loss = criterion(logits, labels_slice)
        if labels.dtype == torch.long:
            hit1 += (logits.topk(1, dim=1)[1] == labels_slice.view(-1, 1)).sum().item()
            hit5 += (logits.topk(5, dim=1)[1] == labels_slice.view(-1, 1)).sum().item()
        loss_value += loss.item() / args.batch_split
        loss_scaler.scale(loss / args.batch_split).backward()

    loss_scaler.step(optimizer)
    loss_scaler.update()
    lr_sched.step()
    batch_ed = datetime.now()

    if i % args.print_freq == 0:
        sync_tensor = torch.tensor([loss_value, hit1 / data.size(0), hit5 / data.size(0)], device="cuda")
        dist.all_reduce(sync_tensor)
        sync_tensor = sync_tensor.cpu() / dist.get_world_size()
        loss_value, acc1, acc5 = sync_tensor.tolist()
        print(
            f"Step {i} batch {(batch_ed - batch_st).total_seconds():.3f}s "
            f"lr {optimizer.param_groups[0]['lr']:.6f} loss {loss_value:.6f} "
            f"acc1 {acc1 * 100:.2f}% acc5 {acc5 * 100:.2f}%"
        )

    if (i + 1) % args.save_freq == 0:
        checkpoint.save_checkpoint(model, optimizer, lr_sched, loss_scaler, i + 1, args)

    if (i + 1) % args.eval_freq == 0:
        print("Eval at step", i + 1)
        model.eval()
        signvlm_main.evaluate(model, val_loader)
        if args.confusion_eval_freq > 0 and (i + 1) % args.confusion_eval_freq == 0:
            yt, yp = collect_predictions(model, val_loader)
            print_confusion_and_f1(yt, yp, NUM_CLASSES, class_names, "validation", save_dir=LIST_DIR)
        model.train()

    batch_st = datetime.now()

checkpoint.save_checkpoint(model, optimizer, lr_sched, loss_scaler, num_steps, args)
print("Training done:", datetime.now() - train_st)

# Train split: confusion matrix + F1 (eval-style loader, no augmentation)
model.eval()
train_eval_loader = make_eval_split_loader(train_tsv, TRAIN_DIR)
yt_tr, yp_tr = collect_predictions(model, train_eval_loader)
print_confusion_and_f1(yt_tr, yp_tr, NUM_CLASSES, class_names, "train", save_dir=LIST_DIR)

1556
Step 0 batch 20.115s lr 0.000040 loss 5.240723 acc1 0.00% acc5 0.00%
Step 10 batch 21.375s lr 0.000040 loss 5.113525 acc1 0.00% acc5 0.00%
Step 20 batch 22.435s lr 0.000040 loss 5.526855 acc1 0.00% acc5 0.00%


KeyboardInterrupt: 

## Cell 8: Evaluation on `test_data` and `Unseen_data`

Runs `main.evaluate` (top-1 / top-5), then **confusion matrix + macro/weighted/micro F1 + per-class report** for **test** and, if `INCLUDE_UNSEEN` from Cell 4, **unseen** (same inference as validation). Latest `checkpoint-*.pth` under `CHECKPOINT_DIR` is loaded unless you change `EVAL_CKPT`.

In [ ]:
# Cell 8: Eval test split and unseen split
import os
from pathlib import Path

import main as signvlm_main


def latest_checkpoint(ckpt_dir):
    files = [f for f in os.listdir(ckpt_dir) if f.startswith("checkpoint-") and f.endswith(".pth")]
    if not files:
        return None
    best = max(files, key=lambda f: int(f.replace("checkpoint-", "").replace(".pth", "")))
    return os.path.join(ckpt_dir, best)


EVAL_CKPT = latest_checkpoint(CHECKPOINT_DIR) or os.path.join(CHECKPOINT_DIR, "checkpoint-10000.pth")

eval_args = argparse.Namespace(
    frames_available=args.frames_available,
    val_list_path=str(test_tsv),
    train_list_path=str(train_tsv),
    train_data_root=TRAIN_DIR,
    val_data_root=TEST_DIR,
    data_root="",
    local_data_cache=getattr(args, "local_data_cache", None) or "",
    batch_size=args.batch_size,
    n_shots=-1,
    num_spatial_views=1,
    num_temporal_views=3,
    num_frames=args.num_frames,
    sampling_rate=args.sampling_rate,
    tsn_sampling=False,
    spatial_size=args.spatial_size,
    mean=args.mean,
    std=args.std,
    num_workers=args.num_workers,
    pin_memory=getattr(args, "pin_memory", True),
    persistent_workers=getattr(args, "persistent_workers", args.num_workers > 0),
        prefetch_factor=getattr(args, "prefetch_factor", 3),
    dummy_dataset=False,
    auto_augment=None,
    interpolation="bicubic",
    mirror=False,
    checkpoint_dir=None,
    auto_resume=False,
    resume_path=None,
    pretrain=None,
)

model.eval()
ckpt = torch.load(EVAL_CKPT, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=True)
print("Loaded:", EVAL_CKPT)

print("=== test_data ===")
test_loader = video_dataset.create_val_loader(eval_args)
signvlm_main.evaluate(model, test_loader)
yt_te, yp_te = collect_predictions(model, test_loader)
print_confusion_and_f1(yt_te, yp_te, NUM_CLASSES, class_names, "test", save_dir=LIST_DIR)

_run_unseen = globals().get("INCLUDE_UNSEEN", False) and os.path.isdir(UNSEEN_DIR)
_ts = Path(unseen_tsv)
if _run_unseen and _ts.is_file() and _ts.stat().st_size > 0:
    eval_args.val_list_path = str(unseen_tsv)
    eval_args.val_data_root = UNSEEN_DIR
    print("=== Unseen_data ===")
    unseen_loader = video_dataset.create_val_loader(eval_args)
    signvlm_main.evaluate(model, unseen_loader)
    yt_un, yp_un = collect_predictions(model, unseen_loader)
    print_confusion_and_f1(yt_un, yp_un, NUM_CLASSES, class_names, "unseen", save_dir=LIST_DIR)
else:
    print("Skipping Unseen_data eval (INCLUDE_UNSEEN=False, missing folder, or empty unseen.tsv).")
